# Export the saved Gemma 2 9B checkpoint-1440 — CPU only

**Run locally in a NEW, SEPARATE notebook kernel. Leave the training notebook and its kernel running.**
This exporter never loads a neural network, imports PyTorch, initializes CUDA, or changes installed packages.
It first copies the saved checkpoint to protect it from checkpoint rotation. Then it packages the pinned cached
base weights, that copied adapter/head, tokenizer, exact preprocessing, validation IDs, and offline Linux wheels.

This is a snapshot of **step 1440**, not the unsaved current in-memory model and not a promise that 1440 will remain best.
No retraining, merging, quantization, calibration, TTA, or ensemble is done here.

Output: **`arena_9b_checkpoint1440_assets.zip`**. Upload it as a **new private Kaggle Dataset**.
Then import **`5_kaggle_9b_checkpoint1440.ipynb`**, attach this dataset and the competition data, select **T4 ×2**, and turn internet off.
The ZIP includes the ~18 GB base weights; it is not a small adapter-only upload.

Keep the original 2B assets and training runs. Do not overwrite them.

In [1]:
from pathlib import Path
import os
import sys

# This applies only to this exporter kernel. Do NOT run this in the training kernel.
if "torch" in sys.modules:
    raise RuntimeError("Open this exporter in a NEW separate kernel; do not use the training kernel.")
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Prefilled from the executed notebook you uploaded.
RUN_DIR = Path(r"C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_9b_20261012_071042")
CHECKPOINT_DIR = RUN_DIR / "checkpoints" / "checkpoint-1440"
# If you ALREADY copied checkpoint-1440 elsewhere, set CHECKPOINT_DIR to that copy instead.
BASE_SNAPSHOT_DIR = None  # Normally resolved from the exact pinned commit in run_config.json.
KAGGLE_PYTHON_VERSIONS = ("3.11", "3.12", "3.13")
EXPORT_PARENT = RUN_DIR.parent

## 1. Protect checkpoint-1440 immediately

This copies only inference files, not the optimizer/RNG state. It does not wait for training to finish.
A missing/deleted/incomplete checkpoint is an error: the exporter never silently substitutes another step.
The copy lives outside `checkpoints/`, so Trainer's retention policy cannot delete it.

In [2]:
import csv
import hashlib
import json
import re
import shutil
import struct
import time
from datetime import datetime


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def tensor_header(path):
    """Check safetensors completeness using its header only; never load large tensors."""
    path = Path(path)
    with path.open("rb") as f:
        raw = f.read(8)
        if len(raw) != 8:
            raise ValueError(f"Incomplete safetensors file: {path}")
        length = struct.unpack("<Q", raw)[0]
        if not 2 <= length <= 100_000_000:
            raise ValueError(f"Invalid safetensors header length: {path}")
        raw_header = f.read(length)
        if len(raw_header) != length:
            raise ValueError(f"Incomplete header: {path}")
        header = json.loads(raw_header)
    tensors = {k:v for k,v in header.items() if k != "__metadata__"}
    end = max((v["data_offsets"][1] for v in tensors.values()), default=0)
    if 8 + length + end != path.stat().st_size:
        raise ValueError(f"Safetensors file is incomplete or has trailing data: {path}")
    return tensors


if not RUN_DIR.is_dir():
    alt = Path("../tmp/outputs") / RUN_DIR.name
    if not alt.is_dir():
        raise FileNotFoundError("Set RUN_DIR to your active arena_9b_20261012_071042 directory.")
    # Only replace CHECKPOINT_DIR if it was the default path.
    if CHECKPOINT_DIR == RUN_DIR / "checkpoints" / "checkpoint-1440":
        CHECKPOINT_DIR = alt / "checkpoints" / "checkpoint-1440"
    RUN_DIR = alt.resolve()
    EXPORT_PARENT = RUN_DIR.parent

for name in ["run_config.json", "preprocessing.json", "tokenization_cases.json", "validation_subsets.csv"]:
    if not (RUN_DIR / name).is_file():
        raise FileNotFoundError(f"Missing run metadata: {RUN_DIR / name}")
for name in ["trainer_state.json", "adapter_config.json", "adapter_model.safetensors"]:
    if not (CHECKPOINT_DIR / name).is_file():
        raise FileNotFoundError(
            f"Missing {CHECKPOINT_DIR / name}. If Trainer deleted checkpoint-1440, "
            "set CHECKPOINT_DIR to your protected copy. A later checkpoint is NOT substituted."
        )
state = read_json(CHECKPOINT_DIR / "trainer_state.json")
if int(state["global_step"]) != 1440:
    raise ValueError(f"Wrong checkpoint step: {state['global_step']} (expected 1440).")
source_file = CHECKPOINT_DIR / "adapter_model.safetensors"
source_header = tensor_header(source_file)
stat_before = (source_file.stat().st_size, source_file.stat().st_mtime_ns)
time.sleep(1)
if stat_before != (source_file.stat().st_size, source_file.stat().st_mtime_ns):
    raise RuntimeError("The checkpoint is still being written. Retry after its save finishes.")

EXPORT_PARENT.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = EXPORT_PARENT / f"export_9b_checkpoint1440_{datetime.now():%Y%m%d_%H%M%S}"
STAGING = EXPORT_DIR / "arena_9b_checkpoint1440_assets"
ADAPTER_COPY = STAGING / "adapter"
ADAPTER_COPY.mkdir(parents=True, exist_ok=False)
for name in ["adapter_model.safetensors", "adapter_config.json", "trainer_state.json",
             "tokenizer.json", "tokenizer.model", "tokenizer_config.json", "special_tokens_map.json",
             "added_tokens.json", "chat_template.jinja"]:
    src = CHECKPOINT_DIR / name
    if src.is_file():
        shutil.copy2(src, ADAPTER_COPY / name)
if (CHECKPOINT_DIR / "chat_templates").is_dir():
    shutil.copytree(CHECKPOINT_DIR / "chat_templates", ADAPTER_COPY / "chat_templates")
for name in ["run_config.json", "preprocessing.json", "tokenization_cases.json", "validation_subsets.csv"]:
    shutil.copy2(RUN_DIR / name, STAGING / name)

# All subsequent work reads this protected copy, not live Trainer checkpoints.
state = read_json(ADAPTER_COPY / "trainer_state.json")
run_config = read_json(STAGING / "run_config.json")
preprocessing = read_json(STAGING / "preprocessing.json")
adapter_config = read_json(ADAPTER_COPY / "adapter_config.json")
header = tensor_header(ADAPTER_COPY / "adapter_model.safetensors")
assert int(state["global_step"]) == 1440
assert header == source_header, "The checkpoint changed during copying."
assert run_config["model_name"] == "google/gemma-2-9b-it"
assert run_config["label_names"] == ["win_a", "win_b", "tie"]
assert preprocessing["version"] == "full-if-fits-v2"
assert (preprocessing["max_length"], preprocessing["prompt_budget"]) == (2048, 384)
assert adapter_config["task_type"] == "SEQ_CLS" and adapter_config["peft_type"] == "LORA"
assert "score" in (adapter_config.get("modules_to_save") or [])
head_keys = [k for k in header if "score" in k and k.endswith("weight")]
assert len(head_keys) == 1 and header[head_keys[0]]["shape"] == [3, 3584]
assert header[head_keys[0]]["dtype"] == "F32", "Expected the original FP32 trained head."
assert any("lora_A" in k for k in header) and any("lora_B" in k for k in header)
evals = [r for r in state.get("log_history", [])
         if int(r.get("step", -1)) == 1440 and "eval_log_loss" in r]
if not evals:
    raise ValueError("No step-1440 validation metric in this checkpoint's trainer_state.json.")
checkpoint_loss = float(evals[-1]["eval_log_loss"])
adapter_hash = sha256_file(ADAPTER_COPY / "adapter_model.safetensors")
print("Checkpoint-1440 is now protected at:", ADAPTER_COPY)
print(f"Recorded SELECTION log loss: {checkpoint_loss:.6f}")
print("Training can keep running. All remaining work is CPU/file I/O only.")

Checkpoint-1440 is now protected at: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\export_9b_checkpoint1440_20261012_180142\arena_9b_checkpoint1440_assets\adapter
Recorded SELECTION log loss: 0.895296
Training can keep running. All remaining work is CPU/file I/O only.


## 2. Resolve the exact cached 9B base model

Only the pinned configuration and weight shards are required. There is no full-snapshot download/check and
no fallback to a newer `main`. Missing optional README/generation files do not invalidate the cache.
The base weights will be streamed directly into the final ZIP, avoiding an additional ~18 GB staging copy.

In [3]:
from huggingface_hub import try_to_load_from_cache

revision = run_config.get("model_revision")
if not isinstance(revision, str) or re.fullmatch(r"[0-9a-f]{40}", revision) is None:
    raise ValueError("The 9B run must have an exact model_revision. Do not guess the base checkpoint.")
if BASE_SNAPSHOT_DIR is None:
    cached = try_to_load_from_cache(run_config["model_name"], "config.json", revision=revision)
    if not isinstance(cached, str) or not Path(cached).is_file():
        raise FileNotFoundError("Pinned config is not cached. Set BASE_SNAPSHOT_DIR to the training snapshot.")
    BASE_SNAPSHOT_DIR = Path(cached).parent
else:
    BASE_SNAPSHOT_DIR = Path(BASE_SNAPSHOT_DIR).expanduser()
if BASE_SNAPSHOT_DIR.name != revision:
    raise ValueError("BASE_SNAPSHOT_DIR must be the snapshot folder named with the training commit.")
base_config = read_json(BASE_SNAPSHOT_DIR / "config.json")
assert base_config["model_type"] == "gemma2" and base_config["hidden_size"] == 3584
assert base_config["num_hidden_layers"] == 42
assert not base_config.get("quantization_config"), "Expected the unquantized training backbone."
index = BASE_SNAPSHOT_DIR / "model.safetensors.index.json"
if index.is_file():
    weight_names = sorted(set(read_json(index)["weight_map"].values()))
    base_names = ["config.json", "model.safetensors.index.json", *weight_names]
else:
    weight_names = ["model.safetensors"]
    base_names = ["config.json", *weight_names]
for name in base_names:
    if Path(name).name != name:
        raise ValueError(f"Unexpected shard path: {name}")
    if not (BASE_SNAPSHOT_DIR / name).is_file():
        raise FileNotFoundError(f"Missing required base shard: {BASE_SNAPSHOT_DIR / name}")
for name in weight_names:
    tensor_header(BASE_SNAPSHOT_DIR / name)

# Trainer normally saves the tokenizer in each checkpoint. Fill absent files only
# from this SAME pinned base snapshot, never from a newer model revision.
for name in ["tokenizer.json", "tokenizer.model", "tokenizer_config.json", "special_tokens_map.json",
             "added_tokens.json", "chat_template.jinja"]:
    if not (ADAPTER_COPY / name).exists() and (BASE_SNAPSHOT_DIR / name).is_file():
        shutil.copy2(BASE_SNAPSHOT_DIR / name, ADAPTER_COPY / name)
for name in ["tokenizer.json", "tokenizer_config.json"]:
    if not (ADAPTER_COPY / name).is_file():
        raise FileNotFoundError(f"Missing saved/cached tokenizer file: {name}")

# Only validation IDs are bundled; no full training CSV or training texts are copied.
subset_rows = list(csv.DictReader((STAGING / "validation_subsets.csv").open(encoding="utf-8", newline="")))
selection_ids = [r["id"] for r in subset_rows if r["subset"] == "selection"]
assert len(selection_ids) == len(set(selection_ids)) and len(selection_ids) > 0
with (STAGING / "selection_ids.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f); writer.writerow(["id"]); writer.writerows([[i] for i in selection_ids])
# Full subset hashes are unnecessary for deployment once selection IDs are recorded.
(STAGING / "validation_subsets.csv").unlink()
manifest = {
    "bundle_format": "arena-gemma2-9b-checkpoint-v1",
    "competition": "llm-classification-finetuning",
    "model_name": run_config["model_name"], "base_revision": revision,
    "checkpoint_step": 1440, "checkpoint_selection_log_loss": checkpoint_loss,
    "selection_rows": len(selection_ids),
    "base_subdir": "base_model", "adapter_subdir": "adapter",
    "adapter_sha256": adapter_hash,
    "training_versions": run_config["versions"],
    "label_names": ["win_a", "win_b", "tie"],
    "label_columns": ["winner_model_a", "winner_model_b", "winner_tie"],
    "head_key": head_keys[0], "head_shape": [3, 3584],
    "attention_backend": "eager", "training_dtype": "bfloat16",
    "quantized": False, "merged": False,
    "inference_recipe": {"swap_tta": False, "temperature": 1.0, "ensemble": False},
    "base_files": {}, "runtimes": {},
}
print("Pinned base:", BASE_SNAPSHOT_DIR)
print("Base bytes:", sum((BASE_SNAPSHOT_DIR / n).stat().st_size for n in base_names))
print("Validation IDs to check on Kaggle:", len(selection_ids))

Pinned base: C:\Users\yana\.cache\huggingface\hub\models--google--gemma-2-9b-it\snapshots\11c9b309abf73637e4b6f9a3fa1e92e615547819
Base bytes: 18483506377
Validation IDs to check on Kaggle: 5747


c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Download offline Linux wheels (no local package changes)

These downloads are for Kaggle, not installations into your active training environment.
The model-library versions must match the 9B run. Kaggle supplies its existing PyTorch/CUDA, NumPy, and pandas.
A Python variant is included only if all its required binary wheels are available. Unsupported variants are reported.
No TorchAO, bitsandbytes, or new GPU driver/CUDA packages are needed.

In [4]:
import subprocess
import warnings
from importlib import metadata
from packaging.markers import default_environment
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

for package in ["transformers", "peft", "accelerate"]:
    if metadata.version(package) != run_config["versions"][package]:
        raise RuntimeError(f"Use the training environment: {package} must be {run_config['versions'][package]}.")
SEED_PACKAGES = ["transformers", "peft", "accelerate", "huggingface_hub", "tokenizers", "safetensors"]
PRESERVE_KAGGLE = {"torch", "numpy", "pandas"}


def locked_requirements(python_version):
    environment = default_environment()
    environment.update({"os_name": "posix", "sys_platform": "linux", "platform_system": "Linux",
        "platform_machine": "x86_64", "python_version": python_version,
        "python_full_version": python_version + ".0", "implementation_version": python_version + ".0", "extra": ""})
    pending, versions, preserved = list(SEED_PACKAGES), {}, set()
    while pending:
        name = canonicalize_name(pending.pop())
        if name in versions or name in PRESERVE_KAGGLE:
            continue
        if name == "triton" or name.startswith("nvidia-") or name in {"torchao", "bitsandbytes"}:
            raise RuntimeError(f"Unexpected optional/GPU dependency {name}; the export will not replace CUDA.")
        installed = metadata.version(name)
        versions[name] = installed
        for item in metadata.requires(name) or []:
            req = Requirement(item)
            if req.marker is not None and not req.marker.evaluate(environment):
                continue
            dep = canonicalize_name(req.name)
            if dep in PRESERVE_KAGGLE:
                preserved.add(req.name + str(req.specifier))
            else:
                if req.specifier and not req.specifier.contains(metadata.version(dep), prereleases=True):
                    raise RuntimeError(f"Inconsistent local dependency: {name} requires {req}.")
                pending.append(dep)
    return [f"{n}=={v}" for n,v in sorted(versions.items())], sorted(preserved)


for py in KAGGLE_PYTHON_VERSIONS:
    wheel_dir = STAGING / "wheels" / py
    wheel_dir.mkdir(parents=True, exist_ok=True)
    requirements, preserved = locked_requirements(py)
    req_file = wheel_dir / "requirements.txt"
    req_file.write_text("\n".join(requirements) + "\n", encoding="utf-8")
    cmd = [sys.executable, "-m", "pip", "download", "--disable-pip-version-check", "--no-deps",
        "--only-binary=:all:", "--platform", "manylinux2014_x86_64", "--platform", "manylinux_2_28_x86_64",
        "--implementation", "cp", "--python-version", py, "--dest", str(wheel_dir), "-r", str(req_file)]
    print(f"Preparing Linux wheels for Python {py}...", flush=True)
    done = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if done.returncode:
        print(done.stdout[-6000:])
        shutil.rmtree(wheel_dir)
        warnings.warn(f"Python {py} variant unavailable; other variants will still be attempted.")
        continue
    manifest["runtimes"][py] = {"wheel_subdir": f"wheels/{py}", "preserved_requirements": preserved}
    print("Prepared", len(list(wheel_dir.glob("*.whl"))), "wheels.")
if not manifest["runtimes"]:
    raise RuntimeError("No Linux runtime prepared. The checkpoint copy is safe; resolve the wheel error before uploading.")
assert "torch" not in sys.modules, "Exporter unexpectedly imported PyTorch."
print("Ready Python runtimes:", list(manifest["runtimes"]))

Preparing Linux wheels for Python 3.11...
Prepared 29 wheels.
Preparing Linux wheels for Python 3.12...
Prepared 29 wheels.
Preparing Linux wheels for Python 3.13...
Prepared 29 wheels.
Ready Python runtimes: ['3.11', '3.12', '3.13']


## 4. Stream the archive and verify file hashes

The large base tensors are read in small blocks directly into an uncompressed ZIP, not loaded into RAM or VRAM.
Allow roughly 20 GB of free disk space for the ZIP (the exact requirement is checked below).
The only model snapshot used is the pinned one from this training run. Checkpoint-1440 was protected in section 1.

In [5]:
import zipfile

archive = EXPORT_DIR / "arena_9b_checkpoint1440_assets.zip"
partial = archive.with_suffix(".zip.partial")
if archive.exists():
    raise FileExistsError(f"Archive already exists: {archive}. Do not overwrite an uploaded version inadvertently.")
base_sources = {f"base_model/{n}": BASE_SNAPSHOT_DIR / n for n in base_names}
for pattern in ["LICENSE*", "NOTICE*", "README.md", "USE_POLICY*"]:
    for p in BASE_SNAPSHOT_DIR.glob(pattern):
        if p.is_file():
            base_sources[f"base_model/{p.name}"] = p
notice = (
    "Private inference bundle: google/gemma-2-9b-it with checkpoint-1440 LoRA and classification head.\n"
    "Subject to Gemma terms: https://ai.google.dev/gemma/terms\n"
    "No training/quantization/merging was performed by this exporter.\n"
)
(STAGING / "MODEL_NOTICE.txt").write_text(notice, encoding="utf-8")
sources = {p.relative_to(STAGING).as_posix(): p for p in STAGING.rglob("*") if p.is_file()}
sources.update(base_sources)
needed = sum(p.stat().st_size for p in sources.values()) + 1024**3
if shutil.disk_usage(EXPORT_DIR).free < needed:
    raise RuntimeError(f"Need about {needed/2**30:.1f} GiB of free disk space for the archive.")
file_hashes = {}
with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as z:
    for rel, src in sorted(sources.items()):
        size = src.stat().st_size
        if size > 1024**3:
            print(f"Archiving {rel}: {size/2**30:.2f} GiB", flush=True)
        h = hashlib.sha256()
        copied = 0
        with src.open("rb") as inp, z.open(f"{STAGING.name}/{rel}", "w", force_zip64=True) as out:
            for block in iter(lambda: inp.read(8 * 1024 * 1024), b""):
                out.write(block); h.update(block); copied += len(block)
        if copied != size:
            raise RuntimeError(f"Source size changed during export: {src}")
        file_hashes[rel] = {"size": size, "sha256": h.hexdigest()}
        if rel.startswith("base_model/") and rel.split("/", 1)[1] in base_names:
            manifest["base_files"][rel.split("/", 1)[1]] = file_hashes[rel]
    manifest["files"] = file_hashes
    assert file_hashes["adapter/adapter_model.safetensors"]["sha256"] == adapter_hash
    manifest_text = json.dumps(manifest, indent=2)
    z.writestr(f"{STAGING.name}/checkpoint_manifest.json", manifest_text)
partial.replace(archive)
(STAGING / "checkpoint_manifest.json").write_text(manifest_text, encoding="utf-8")
print("\nUPLOAD AS A NEW PRIVATE KAGGLE DATASET:")
print(archive.resolve())
print(f"Size: {archive.stat().st_size/2**30:.2f} GiB")
print("Then import 5_kaggle_9b_checkpoint1440.ipynb; attach ONLY this model bundle plus competition data.")
print("No change was made to the running training model, its checkpoints, or installed packages.")

Archiving base_model/model-00001-of-00004.safetensors: 4.57 GiB
Archiving base_model/model-00002-of-00004.safetensors: 4.61 GiB
Archiving base_model/model-00003-of-00004.safetensors: 4.62 GiB
Archiving base_model/model-00004-of-00004.safetensors: 3.42 GiB

UPLOAD AS A NEW PRIVATE KAGGLE DATASET:
C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\export_9b_checkpoint1440_20261012_180142\arena_9b_checkpoint1440_assets.zip
Size: 17.73 GiB
Then import 5_kaggle_9b_checkpoint1440.ipynb; attach ONLY this model bundle plus competition data.
No change was made to the running training model, its checkpoints, or installed packages.


## References and limitations

The source is your executed `3_arena_gemma2_9b_memory_safe(1).ipynb`, run `arena_9b_20261012_071042`.
Only a saved checkpoint can be exported concurrently; unsaved live GPU weights are not read.

[PEFT checkpoint format](https://huggingface.co/docs/peft/developer_guides/checkpoint)
· [pip cross-platform wheel downloads](https://pip.pypa.io/en/stable/cli/pip_download/)
· [competition requirements](https://www.kaggle.com/competitions/llm-classification-finetuning)

This notebook has been syntax-checked and its file operations tested with synthetic checkpoints.
The actual local weights are not available in the assistant environment. Full-model export and Kaggle inference require your files.